In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from starmap.config import Config
from starmap.pipeline import Pipeline
import starmap.io as io
import starmap.preprocessing as preproc
import starmap.gene_calling as genecall
import starmap.cell_segmentation as cellseg
from cellpose import utils
import os
import pandas as pd
from datetime import datetime
import h5py
import tifffile
import copy

import IPython.display
import PIL
from PIL import Image, ImageDraw

import bardensr
import bardensr.plotting

def create_codebook(filepath_codebook, round_num, base_code): # working with round index
    genenames, gene_codes = [], []
    with open(filepath_codebook, newline='') as f:
        reader = csv.reader(f)
        for row in reader:
            check_head = set(row[1])
            available = set('ATGC')
            if check_head.issubset(available):
                if row[0].__contains__(codecs.BOM_UTF8.decode(f.encoding)):
                # A Byte Order Mark is present
                    genenames.append(row[0].strip(codecs.BOM_UTF8.decode(f.encoding)))
                else:
                    genenames.append(row[0])
                barcode = np.array([*row[1]])
                # print(barcode)
                barcode_subset = barcode[round_num]
                # print(barcode_subset)
                gene_codes.append(barcode_subset)
    print(datetime.now().strftime("%d/%m/%Y %H:%M:%S"), "        "+"genenames: ", genenames)
    print(datetime.now().strftime("%d/%m/%Y %H:%M:%S"), "        "+"gene_codes: ", gene_codes)

    codebook = np.full((len(round_num), 4, len(gene_codes)), False)
    for tagNum, tag in enumerate(gene_codes):
        for roundNum, letter in enumerate(tag):
            if letter == base_code[0]: 
                codebook[roundNum, 0, tagNum] = True
            if letter == base_code[1]: 
                codebook[roundNum, 1, tagNum] = True
            if letter == base_code[2]: 
                codebook[roundNum, 2, tagNum] = True
            if letter == base_code[3]: 
                codebook[roundNum, 3, tagNum] = True
    codeflat = codebook.reshape((codebook.shape[0]*codebook.shape[1],-1)) # R,C,J=codebook.shape
    return codebook, genenames, codeflat

def open_hdf5(file_path): #assumes this is a list of np arrays
    var_name= []
    f = h5py.File(file_path)
    print(f)
    data = f['list']
    for i in range(len(data)):
        print(i)
        var_name.append(data[str(i)][:])
    f.close()
    return var_name


import numpy as np
from skimage.transform import resize
def open_merged_fov_files(file_path, FOVs, file_type):
    all_fov_data = []  # List to store all FOV data
    
    for FOV in FOVs:
        # Open the hdf5 file for the current FOV
        fov_data = open_hdf5(f"{file_path}fov{FOV}{file_type}.hdf5")
        
        # Ensure fov_data is a numpy array (if it's a list, convert it)
        if isinstance(fov_data, list):
            fov_data = np.array(fov_data)  # Convert the list to a numpy array
            
        fov_resized = []  # List to hold resized data for this FOV
        
        # Iterate over the channels (6), then slices (4)
        for c in range(fov_data.shape[0]):  # Loop through channels (6)
            for s in range(fov_data.shape[1]):  # Loop through slices (4)
                # Resize each slice (channel, slice) to (2304, 2304)
                fov_data_resized = resize(fov_data[c, s], (2304, 2304), mode='constant', preserve_range=True)
                fov_resized.append(fov_data_resized)  # Append the resized slice to fov_resized
        
        # Convert the resized list to a numpy array and reshape it back to the correct dimensions (6, 4, 2304, 2304)
        fov_resized = np.array(fov_resized).reshape(fov_data.shape[0], fov_data.shape[1], 2304, 2304)
        
        # Append the resized FOV data to the all_fov_data list
        all_fov_data.append(fov_resized)
    
    # Stack all FOVs into a single numpy array along the first axis (FOV dimension)
    all_fov_data = np.stack(all_fov_data, axis=0)
    
    return all_fov_data

def save_hdf5(matrix, file_path): #assumes this is a list of np arrays
    f = h5py.File(file_path,'w')
    grp = f.create_group('list')
    for i,arr in enumerate(matrix):
        grp.create_dataset(str(i), data=arr)
    f.close()
    return None

Load image file, align channels, color correct, and align rounds 

In [ ]:
path = "/home/dfaltin1/data_jkebsch1/DFaltine/Mouse_Actin/"

In [ ]:
image_origin = open_hdf5(path + 'image_origin_2DF08_Actin.hdf5')

In [ ]:
# check aligned images ALL
IPython.display.Image(bardensr.plotting.gify(image_origin[0][:,:,:, :],normstyle='each', duration=500),width=500)

In [ ]:
from skimage.feature import blob_dog, blob_log, blob_doh
import cv2

def blob_keypoints(im):
	blobs = sk.feature.blob_dog(
		im,
		# min_sigma=3,  # Minimum size of blobs
		max_sigma=10,  # Maximum size of blobs
		# num_sigma=10,  # Number of intermediate sizes
		threshold=0.05  # Lower threshold detects fainter blobs
	)
	keypoints = []
	for blob in blobs:
		y, x, sigma = blob
		keypoints.append(cv2.KeyPoint(float(x), float(y), float(2 * sigma)))
	return keypoints


def channel_alignment_get_transforms_opencv_sm(images):  # assumes structure is round x channel x X x Y
	transforms = []
	images = np.stack(images, axis=0)  # Ensure this is a NumPy array with the right shape
	print("Shape of images after stacking:", images.shape)
	for i in range(len(images)):
		print("Finding channel alignment for round " + str(i))
		print(images.shape)
		 
		#Use correct indexing
		#Use correct indexing
		c2 = images[i][0]  # Assuming this is the intended channel
		c2 = np.expand_dims(np.stack([c2, c2, c2, c2], axis=0), axis=1)
		print(c2.shape)
		c2 = bardensr.preprocessing.background_subtraction((c2 / np.max(c2)).astype('float'), [0, 50, 50])
		c2 = np.sum(c2[:,0], axis=0)
		c2 = np.clip(c2, np.percentile(c2.ravel(), 0), np.percentile(c2.ravel(), 98)) # moved clipping here
		c2 = ((c2 / np.max(c2)) * 250).astype('uint8') 


		c3 = images[i][1]  # Correct indexing
		c3 = np.expand_dims(np.stack([c3, c3, c3, c3], axis=0), axis=1)
		print(c3.shape)
		c3 = bardensr.preprocessing.background_subtraction((c3 / np.max(c3)).astype('float'), [0, 50, 50])
		c3 = np.sum(c3[:,0], axis=0)
		c3 = np.clip(c3, 0, np.percentile(c3.ravel(), 98))
		c3 = ((c3 / np.max(c3)) * 250).astype('uint8')
		
		blobs1 = blob_log(c3, max_sigma=10, threshold=0.1)
		blobs2 = blob_log(c2, max_sigma=10, threshold=0.1)

		# Extract keypoints' coordinates
		keypts1 = np.array([[b[1], b[0]] for b in blobs1], dtype=np.float32)
		keypts2 = np.array([[b[1], b[0]] for b in blobs2], dtype=np.float32)
		
		sift = cv2.SIFT_create()
		
		keypts1_pt = [cv2.KeyPoint(x=pt[0], y=pt[1], size=10) for pt in keypts1]
		keypts2_pt = [cv2.KeyPoint(x=pt[0], y=pt[1], size=10) for pt in keypts2]
		keypoints1, descriptors1 = sift.compute(c3, keypts1_pt)
		keypoints2, descriptors2 = sift.compute(c2, keypts2_pt)

		# Need to draw only good matches, so create a mask
		bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)  # Use NORM_HAMMING for ORB
		# Find the two nearest matches for each descriptor
		matches = bf.knnMatch(descriptors1, descriptors2, k=1, mask = None)
		#matchesMask = [[0,0] for i in range(len(matches))]
		# ratio test as per Lowe's paper
		src_pts = []
		dst_pts = []
		print(matches)
		matches = [match for match in matches if len(match) > 0]
		for match in matches:
		# Since k=1, each match contains only one DMatch object
			m = match[0]  # Access the first (and only) match
			src_pts.append(keypoints1[m.queryIdx].pt)  # Get the source point from the queryIdx
			dst_pts.append(keypoints2[m.trainIdx].pt)  # Get the destination point from the trainIdx
		# Convert points to numpy arrays
		src_pts = np.float32(src_pts).reshape(-1, 1, 2)
		dst_pts = np.float32(dst_pts).reshape(-1, 1, 2)
		# Find homography
		H, _ = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 0.5)
		transforms.append(H)
	# Save outside of the loop to avoid overwriting
	save_hdf5(transforms, path + "/transforms.hdf5")

In [ ]:
fov0 = image_origin[0]

In [ ]:
channel_alignment_get_transforms_opencv_sm(fov0)

In [ ]:
colormixing_matrix = np.array([
    [1,0.63,0,0],  # matrix[0,1] is positive because whenever frame 0 is bright, frame 1 is also a bit bright
    [0.4,1,0,0],
    [0,0,1,0.40],
    [0,0,0.65,1]])

In [ ]:
import cv2

def get_border_size(image, x_dim, y_dim):
    data_temp = image.reshape(image.shape[0]*image.shape[1], x_dim, y_dim)
    mask = np.ones([x_dim, y_dim])

    for i in range(len(data_temp)):
        mask = mask*data_temp[i] != 0

    nonzero_coords = np.argwhere(mask != 0)
    top, left = nonzero_coords.min(axis=0)
    bottom, right = nonzero_coords.max(axis=0)

    cropped_image = data_temp[:,top+20:bottom-20, left+20:right-20].astype(np.float64)
    mask_fov = np.zeros([x_dim, y_dim])
    mask_fov[top+20:bottom-20, left+20:right-20] = 1 # clip 20 pixels on each side to account for rotation
    data_temp = cropped_image.reshape(image.shape[0],image.shape[1],cropped_image.shape[1],cropped_image.shape[2])
    return data_temp, mask_fov

def alignment_opencv_colorbleed_corr_crop_sm(images, transforms, fov_sample, colormixing_matrix=colormixing_matrix): # assumes structure is fov x round x channel x X x Y
	# Apply channel alignment
	from skimage.feature import blob_dog, blob_log, blob_doh

	for k in range(len(transforms)):
			print(images[k][1].shape)
			images[k][1] = cv2.warpPerspective((images[k][1]).astype(np.float32), transforms[k], (2304, 2304))
			images[k][3] = cv2.warpPerspective(images[k][3].astype(np.float32), transforms[k], (2304, 2304))
	transforms = []
	r1 = np.expand_dims(images[0],axis = 1)
	r1 = bardensr.preprocessing.background_subtraction((r1/np.max(r1)).astype('float'),[0,30,30])
	r1 = np.sum(r1[:,0], axis = 0)
	r1 = np.clip(r1, np.percentile(r1.ravel(), 0), np.percentile(r1.ravel(), 95))
	r1 = ((r1/np.max(r1))*255).astype('uint8')
	r11 = r1 #[400:1700, 400:1700]
	temp = []
	for j in range(len(images)):
			print("round alignment for fov "+str(j)+" and round "+ str(j))
			rn = np.expand_dims(images[j],axis = 1)
			rn = bardensr.preprocessing.background_subtraction((rn/np.max(rn)).astype('float'),[0,30,30])
			rn = np.sum(rn[:,0], axis = 0)
			print(rn.shape)
			rn = np.clip(rn, np.percentile(rn.ravel(), 1), np.percentile(rn.ravel(), 95))
			rn = ((rn/np.max(rn))*255).astype('uint8')
			rn1 = rn #[400:1700, 400:1700]
			# Detect blobs
			# Blob detection
			blobs1 = blob_log(rn1, max_sigma=10, threshold=0.1)
			blobs2 = blob_log(r11, max_sigma=10, threshold=0.1)
			# Extract keypoints' coordinates
			keypts1 = np.array([[b[1], b[0]] for b in blobs1], dtype=np.float32)
			keypts2 = np.array([[b[1], b[0]] for b in blobs2], dtype=np.float32)
			sift = cv2.SIFT_create()
			keypts1_pt = [cv2.KeyPoint(x=pt[0], y=pt[1], size=10) for pt in keypts1]
			keypts2_pt = [cv2.KeyPoint(x=pt[0], y=pt[1], size=10) for pt in keypts2]
			keypoints1, descriptors1 = sift.compute(rn, keypts1_pt)
			keypoints2, descriptors2 = sift.compute(r1, keypts2_pt)
			#print(des1,des2)
			# FLANN parameters
			FLANN_INDEX_KDTREE = 1
			index_params = dict(algorithm = FLANN_INDEX_KDTREE, trees = 5)
			search_params = dict(checks=50) # or pass empty dictionary
			flann = cv2.FlannBasedMatcher(index_params,search_params)
			matches = flann.knnMatch(descriptors1,descriptors2,k=2)
			# Need to draw only good matches, so create a mask
			#matchesMask = [[0,0] for i in range(len(matches))]
			# ratio test as per Lowe's paper
			good = []
			for p,(m,n) in enumerate(matches):
				if m.distance < 0.95*n.distance:
					good.append(m)
			# Sort matches by score
			#matches = sorted(matches, key=lambda x: x.distance)
			print('Number of matches between round 1 and ' + str(j)+": "+ str(len(good)))
			#img3 = cv2.drawMatches(r11,kp1,rn1,kp2,good[:5],None,flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
			#plt.imshow(img3),plt.show()
			# Extract matched keypoints
			src_pts1 = np.float32([keypoints1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
			dst_pts1 = np.float32([keypoints2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
			# Find homography
			H, _ = cv2.estimateAffinePartial2D(src_pts1, dst_pts1,method=cv2.RANSAC, ransacReprojThreshold=1)
			H = np.asarray(H, dtype=np.float32)
			print(H)
			#temp[j] = H
			images[ j][0] = cv2.warpAffine(images[ j][ 0], H, (images[ j][ 0].shape[1], images[ j][ 0].shape[0]))
			images[ j][ 1] = cv2.warpAffine(images[ j][ 1], H, (images[ j][ 1].shape[1], images[ j][ 1].shape[0]))
			images[ j][ 2] = cv2.warpAffine(images[ j][ 2], H, (images[ j][ 2].shape[1], images[ j][ 2].shape[0]))
			images[ j][ 3] = cv2.warpAffine(images[ j][ 3], H, (images[ j][ 3].shape[1], images[ j][ 3].shape[0]))  
	fix=np.linalg.inv(colormixing_matrix)
	for i in range(len(images)):
		print("Round: "+str(i))
		images[i] = np.clip(np.einsum('cxy,cd->dxy',images[i],fix),0,None)
	print("crop images and find mask")
	#crop image
	images_crop = []
	mask = []
	# Find the maximum border size among all images
	images = np.stack(images, axis=0)
	print("got the images")
	images_crop,mask = get_border_size(images, 2304, 2304)
	print("got the crops")

	return images, images_crop, mask

In [ ]:
transforms = open_hdf5(path + 'transforms.hdf5')

In [ ]:
alignedfov0, alignedfov0_crop, aligned_fov0_mask  = alignment_opencv_colorbleed_corr_crop_sm(fov0, transforms, fov_sample = 0)
save_hdf5(alignedfov0, path + "fov0_align_corr.hdf5")

In [ ]:
IPython.display.Image(bardensr.plotting.gify(alignedfov0_crop[:,:,:, :],normstyle='each', duration=500),width=500)

In [ ]:
def register_stack_with_demons(
	imagestack,
	reference_index=0,
	niter=150,
	smoothing_sigma=2.0,
):
	"""
	Demon registration on full-resolution images (no downsampling)
	"""
	R, C, H, W = imagestack.shape
	images_sum = np.sum(imagestack, axis=1)

	# Prepare fixed SITK image
	fixed_np = images_sum[reference_index]
	fixed = to_sitk_image(fixed_np)
	transforms = [None]*R
	aligned_stack = np.zeros_like(imagestack)

	# Reference frame identity
	aligned_stack[reference_index] = imagestack[reference_index]
	identity_tf = sitk.Transform(2, sitk.sitkIdentity)
	transforms[reference_index] = identity_tf

	for i in range(R):
		if i == reference_index:
			continue
		print(f"Processing frame {i}")
		moving_np = images_sum[i]
		moving = to_sitk_image(moving_np)

		# Initialize and run Demons registration
		demons = sitk.DemonsRegistrationFilter()
		demons.SetNumberOfIterations(niter)
		demons.SetStandardDeviations(smoothing_sigma)

		displacement_field = demons.Execute(fixed, moving)
		print(f"RMS Change: {demons.GetRMSChange():.6f} after {demons.GetElapsedIterations()} iterations")

		# Convert to displacement field transform
		displacement_field = sitk.DisplacementFieldTransform(displacement_field)
		transforms[i] = displacement_field

		# Resample all channels
		for ch in range(C):
			mov_ch = to_sitk_image(imagestack[i,ch])
			res = sitk.Resample(
				mov_ch, fixed, displacement_field,
				sitk.sitkLinear, 0.0, mov_ch.GetPixelID()
			)
			aligned_stack[i,ch] = from_sitk_image(res)
	return transforms, aligned_stack

def apply_mask_to_images(image_array, mask):
	mask = np.stack(mask, axis = 0)
	mask_width, mask_height = mask.shape
	x_index, y_index = np.where(mask == 1)
	x_st, x_ed, y_st, y_ed = min(x_index), max(x_index), min(y_index), max(y_index)
	result_array = np.zeros((image_array.shape[0], image_array.shape[1], mask_width, mask_height), dtype=image_array.dtype)

	for i in range(image_array.shape[0]):
		for j in range(image_array.shape[1]):
			x_core, y_core = image_array[i, j].shape[0], image_array[i, j].shape[1]
			#if (x_ed-x_st+1) != x_core or (y_ed-y_st+1) != y_core:
			#print(datetime.now().strftime("%d/%m/%Y %H:%M:%S"), "        "+"masks not fit!")
			resized_image = np.zeros((mask_width, mask_height), dtype=image_array.dtype)
			resized_image[x_st:x_ed+1, y_st:y_ed+1] = image_array[i, j]
			result_array[i, j] = resized_image
	return result_array

def to_sitk_image(arr):
	if arr.ndim == 2:
		return sitk.GetImageFromArray(arr)
	elif arr.ndim == 3:
		return sitk.GetImageFromArray(arr, isVector=False)
	else:
		raise ValueError("Only 2D or 3D arrays are supported.")

def from_sitk_image(img):
	return sitk.GetArrayFromImage(img)

In [ ]:
# Demons Registration
import SimpleITK as sitk

images = alignedfov0_crop
transforms, aligned_stack = register_stack_with_demons(images)
image_reg = apply_mask_to_images(aligned_stack, aligned_fov0_mask)

In [ ]:
IPython.display.Image(bardensr.plotting.gify(image_reg[:,:,:, :],normstyle='each', duration=500),width=500)

In [ ]:
# ============================================================
# FULL SCRIPT: Detect blobs -> track across rounds -> extract peak per channel
# -> robust normalization from DETECTED peaks (handles outliers + background channels)
# -> dominant channel per blob/round + switching
#
# Expected input:
#   images_aligned: numpy array shaped (R, C, H, W)
#     R = rounds, C = channels, H/W = image size
# ============================================================

import builtins
import numpy as np
import pandas as pd
from skimage.feature import blob_log
from scipy.optimize import linear_sum_assignment

# ----------------------------
# Detection helpers
# ----------------------------

def make_reference_image(images_r: np.ndarray, clip=(1, 99)) -> np.ndarray:
    """
    images_r: (C,H,W) for a single round. Returns 2D float image for blob detection.
    Uses sum across channels + percentile clipping + 0-1 scaling.
    """
    img = np.sum(images_r.astype(np.float32), axis=0)
    lo, hi = np.percentile(img.ravel(), clip[0]), np.percentile(img.ravel(), clip[1])
    img = np.clip(img, lo, hi)
    if img.max() > 0:
        img = img / img.max()
    return img


def detect_blobs_per_round(
    images: np.ndarray,
    min_sigma=1.2,
    max_sigma=5.0,
    num_sigma=15,
    threshold=0.04,
    overlap=0.2,
    clip=(5, 99.5),
) -> pd.DataFrame:
    """
    images: (R,C,H,W)
    Returns DataFrame with columns: round, det_id, x, y, sigma, r
    """
    if isinstance(images, list):
        images = np.stack(images, axis=0)

    R, C, H, W = images.shape
    rows = []

    for r in range(R):
        ref = make_reference_image(images[r], clip=clip)
        blobs = blob_log(
            ref,
            min_sigma=min_sigma,
            max_sigma=max_sigma,
            num_sigma=num_sigma,
            threshold=threshold,
            overlap=overlap,
        )
        # blob_log returns (y, x, sigma)
        for det_id, (yy, xx, sig) in enumerate(blobs):
            rr = int(max(2, 3.0 * float(sig)))  # ROI radius
            rows.append({
                "round": r,
                "det_id": det_id,
                "x": float(xx),
                "y": float(yy),
                "sigma": float(sig),
                "r": rr,
            })

    return pd.DataFrame(rows)


# ----------------------------
# Tracking helpers
# ----------------------------

def _build_cost_matrix(prev: pd.DataFrame, curr: pd.DataFrame, w_dist=1.0, w_size=0.5) -> np.ndarray:
    """
    Cost = w_dist * euclidean_distance + w_size * abs(sigma_diff)
    """
    prev_xy = prev[["x", "y"]].to_numpy(dtype=np.float32)
    curr_xy = curr[["x", "y"]].to_numpy(dtype=np.float32)

    dx = prev_xy[:, None, 0] - curr_xy[None, :, 0]
    dy = prev_xy[:, None, 1] - curr_xy[None, :, 1]
    dist = np.sqrt(dx * dx + dy * dy)

    prev_sig = prev["sigma"].to_numpy(dtype=np.float32)[:, None]
    curr_sig = curr["sigma"].to_numpy(dtype=np.float32)[None, :]
    size = np.abs(prev_sig - curr_sig)

    return w_dist * dist + w_size * size


def track_blobs_across_rounds(
    detections: pd.DataFrame,
    max_dist_px=4.0,
    max_cost=6.0,
    w_dist=1.0,
    w_size=0.5,
    allow_gap=0,
) -> pd.DataFrame:
    """
    Assigns persistent blob_id across rounds using Hungarian assignment.
    Returns detections with added column blob_id.
    """
    det = detections.copy()
    det["blob_id"] = -1

    by_round = {r: df.reset_index(drop=True) for r, df in det.groupby("round", sort=True)}
    rounds = sorted(by_round.keys())
    if not rounds:
        return det

    next_id = 0
    active = {}  # blob_id -> {x,y,sigma,last_round}

    # initialize first round
    r0 = rounds[0]
    first = by_round[r0].copy()
    first_ids = np.arange(next_id, next_id + len(first), dtype=int)
    first["blob_id"] = first_ids
    next_id += len(first)

    for _, row in first.iterrows():
        active[int(row["blob_id"])] = {
            "x": float(row["x"]), "y": float(row["y"]), "sigma": float(row["sigma"]),
            "last_round": int(row["round"])
        }
    by_round[r0] = first

    # subsequent rounds
    for r in rounds[1:]:
        curr = by_round[r].copy()

        # candidates from active within allow_gap
        candidates = []
        cand_ids = []
        for bid, st in active.items():
            if (r - st["last_round"]) <= (allow_gap + 1):
                candidates.append(st)
                cand_ids.append(bid)

        if len(curr) == 0:
            by_round[r] = curr
            continue

        if len(candidates) == 0:
            new_ids = np.arange(next_id, next_id + len(curr), dtype=int)
            curr["blob_id"] = new_ids
            next_id += len(curr)
            for _, row in curr.iterrows():
                active[int(row["blob_id"])] = {
                    "x": float(row["x"]), "y": float(row["y"]), "sigma": float(row["sigma"]),
                    "last_round": int(row["round"])
                }
            by_round[r] = curr
            continue

        prev_df = pd.DataFrame(candidates)
        prev_df["blob_id"] = cand_ids

        # gating distance
        prev_xy = prev_df[["x", "y"]].to_numpy(dtype=np.float32)
        curr_xy = curr[["x", "y"]].to_numpy(dtype=np.float32)
        dx = prev_xy[:, None, 0] - curr_xy[None, :, 0]
        dy = prev_xy[:, None, 1] - curr_xy[None, :, 1]
        dist = np.sqrt(dx * dx + dy * dy)

        cost = _build_cost_matrix(prev_df, curr, w_dist=w_dist, w_size=w_size)

        HUGE = 1e9
        cost = np.where(dist <= max_dist_px, cost, HUGE)

        row_ind, col_ind = linear_sum_assignment(cost)

        for i, j in zip(row_ind, col_ind):
            if cost[i, j] >= HUGE:
                continue
            if cost[i, j] > max_cost:
                continue
            bid = int(prev_df.loc[i, "blob_id"])
            curr.loc[j, "blob_id"] = bid

        # new tracks for unassigned
        unassigned = curr["blob_id"] == -1
        n_new = int(unassigned.sum())
        if n_new > 0:
            new_ids = np.arange(next_id, next_id + n_new, dtype=int)
            curr.loc[unassigned, "blob_id"] = new_ids
            next_id += n_new

        # update active
        for _, row in curr.iterrows():
            bid = int(row["blob_id"])
            active[bid] = {
                "x": float(row["x"]), "y": float(row["y"]), "sigma": float(row["sigma"]),
                "last_round": int(row["round"])
            }

        by_round[r] = curr

        # prune old tracks
        to_del = [bid for bid, st in active.items() if (r - st["last_round"]) > (allow_gap + 1)]
        for bid in to_del:
            del active[bid]

    return pd.concat([by_round[r] for r in rounds], ignore_index=True)


# ----------------------------
# ROI + peak extraction
# ----------------------------

def extract_roi(image2d: np.ndarray, x: float, y: float, r: int):
    x = int(builtins.round(x))
    y = int(builtins.round(y))
    r = int(r)

    H, W = image2d.shape
    x0 = builtins.max(0, x - r)
    x1 = builtins.min(W, x + r + 1)
    y0 = builtins.max(0, y - r)
    y1 = builtins.min(H, y + r + 1)

    return image2d[y0:y1, x0:x1], (x0, y0)


def per_blob_analysis_brightest_pixel(images: np.ndarray, tracked_detections: pd.DataFrame) -> pd.DataFrame:
    """
    For each (blob_id, round, channel), find the brightest pixel in an ROI around detection center.
    Returns: blob_id, round, channel, x_peak, y_peak, intensity_peak
    """
    if isinstance(images, list):
        images = np.stack(images, axis=0)

    R, C, H, W = images.shape
    rows = []

    for row in tracked_detections.itertuples(index=False):
        r = int(row.round)
        bid = int(row.blob_id)
        x = float(row.x); y = float(row.y)
        rr = int(row.r)

        for ch in range(C):
            roi, (x0, y0) = extract_roi(images[r, ch], x, y, rr)
            if roi.size == 0:
                continue
            idx = int(np.argmax(roi))
            yy, xx = np.unravel_index(idx, roi.shape)
            rows.append({
                "blob_id": bid,
                "round": r,
                "channel": ch,
                "x_peak": x0 + xx,
                "y_peak": y0 + yy,
                "intensity_peak": float(roi[yy, xx]),
            })

    return pd.DataFrame(rows)


# ----------------------------
# Robust normalization from detected peaks (handles outliers + background channels)
# ----------------------------

def normalize_peaks_robust_ignore_bad_channels(
    peaks_df: pd.DataFrame,
    q: float = 99.5,
    q_lo: float = 50.0,
    min_ratio: float = 1.5,
    min_scale: float = None,
    eps: float = 1e-12,
    clamp: bool = True,
):
    if peaks_df.empty:
        out = peaks_df.copy()
        out["intensity_norm"] = np.nan
        out["bad_channel"] = False
        return out, pd.DataFrame()

    g = peaks_df.groupby("channel")["intensity_peak"]

    ch_stats = pd.DataFrame({
        "scale_hi": g.quantile(q / 100.0),
        "baseline": g.quantile(q_lo / 100.0),
        "n": g.size(),  # <-- FIX: call the method
    }).astype({"n": int})

    ch_stats["ratio"] = ch_stats["scale_hi"] / np.maximum(ch_stats["baseline"], eps)

    bad = ch_stats["ratio"] < min_ratio
    if min_scale is not None:
        bad = bad | (ch_stats["scale_hi"] < float(min_scale))
    ch_stats["bad_channel"] = bad

    scale = peaks_df["channel"].map(ch_stats["scale_hi"]).to_numpy(np.float32)
    scale = np.where(scale > eps, scale, 1.0)

    raw = peaks_df["intensity_peak"].to_numpy(np.float32)
    if clamp:
        raw = np.minimum(raw, scale)

    norm = raw / scale

    bad_mask = peaks_df["channel"].map(ch_stats["bad_channel"]).fillna(True).to_numpy(bool)
    norm = np.where(bad_mask, 0.0, norm)

    peaks_norm = peaks_df.copy()
    peaks_norm["intensity_norm"] = norm.astype(np.float32)
    peaks_norm["bad_channel"] = bad_mask

    return peaks_norm, ch_stats



# ----------------------------
# Dominant channel + switching
# ----------------------------

def summarize_channel_switching(peaks_df: pd.DataFrame, use_col: str = "intensity_norm") -> pd.DataFrame:
    """
    For each (blob_id, round), pick channel with max intensity (raw or normalized).
    Adds prev_channel and switched boolean.
    """
    dom = (
        peaks_df.sort_values(use_col, ascending=False)
                .groupby(["blob_id", "round"], as_index=False)
                .first()
    )
    dom = dom.sort_values(["blob_id", "round"])
    dom["prev_channel"] = dom.groupby("blob_id")["channel"].shift(1)
    dom["switched"] = dom["prev_channel"].notna() & (dom["channel"] != dom["prev_channel"])
    return dom


# ----------------------------
# END-TO-END DRIVER
# ----------------------------

def run_blob_pipeline(
    images_aligned: np.ndarray,
    # detection
    min_sigma=2.0, max_sigma=10.0, num_sigma=10, threshold=0.1, overlap=0.5,
    # tracking
    max_dist_px=4.0, max_cost=6.0, w_dist=1.0, w_size=0.5, allow_gap=0,
    # normalization
    norm_q=99.5, norm_min_ratio=1.5, norm_min_scale=None, norm_clamp=True,
):
    """
    Returns:
      detections_df: per-round detections
      tracked_df: detections with blob_id assigned across rounds
      peaks_df: per blob/round/channel peaks with intensity_norm + bad_channel
      dominant_df: dominant channel per blob/round based on intensity_norm
      ch_stats: per-channel diagnostics (scale_hi, baseline, ratio, bad_channel, n)
    """
    det = detect_blobs_per_round(
        images_aligned,
        min_sigma=min_sigma,
        max_sigma=max_sigma,
        num_sigma=num_sigma,
        threshold=threshold,
        overlap=overlap,
    )

    tracked = track_blobs_across_rounds(
        det,
        max_dist_px=max_dist_px,
        max_cost=max_cost,
        w_dist=w_dist,
        w_size=w_size,
        allow_gap=allow_gap,
    )

    peaks = per_blob_analysis_brightest_pixel(images_aligned, tracked)

    peaks_norm, ch_stats = normalize_peaks_robust_ignore_bad_channels(
        peaks,
        q=norm_q,
        min_ratio=norm_min_ratio,
        min_scale=norm_min_scale,
        clamp=norm_clamp,
    )

    dominant = summarize_channel_switching(peaks_norm, use_col="intensity_norm")

    return det, tracked, peaks_norm, dominant, ch_stats

In [ ]:
detections_df, tracked_df, peaks_df, dominant_df, ch_stats = run_blob_pipeline(
    images_aligned=image_reg,
    threshold=0.04,
    max_dist_px=10.0,
    allow_gap=0,
    norm_q=99.5,
    norm_min_ratio=1.5,
    norm_min_scale=150,
    norm_clamp=True
)



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def merged_round_image(images, r, clip=(1, 99)):
    # images: (R,C,H,W)
    img = images[r].sum(axis=0).astype(np.float32)
    lo, hi = np.percentile(img, clip[0]), np.percentile(img, clip[1])
    img = np.clip(img, lo, hi)
    img /= (img.max() + 1e-9)
    return img

r = 0
bg = merged_round_image(image_reg, r)
life = tracked_df.groupby("blob_id")["round"].nunique()

good_ids3 = life[life >= 0].index
d3 = tracked_df[(tracked_df["round"] == r) & (tracked_df["blob_id"].isin(good_ids3))]

plt.figure(figsize=(10,10))
plt.imshow(bg, cmap="gray")
plt.scatter(d3["x"], d3["y"], s=0.1, facecolors="none", edgecolors="red")
plt.title(f"Tracked blobs overlay (round {r})")
plt.axis("off")
plt.show()


In [ ]:
r = 1
bg = merged_round_image(image_reg, r)
life = tracked_df.groupby("blob_id")["round"].nunique()

good_ids3 = life[life >= 0].index
d3 = tracked_df[(tracked_df["round"] == r) & (tracked_df["blob_id"].isin(good_ids3))]

plt.figure(figsize=(10,10))
plt.imshow(bg, cmap="gray")
plt.scatter(d3["x"], d3["y"], s=0.1, facecolors="none", edgecolors="red")
plt.title(f"Tracked blobs overlay (round {r})")
plt.axis("off")
plt.show()

In [ ]:
r = 2
bg = merged_round_image(image_reg, r)
life = tracked_df.groupby("blob_id")["round"].nunique()

good_ids3 = life[life >= 0].index
d3 = tracked_df[(tracked_df["round"] == r) & (tracked_df["blob_id"].isin(good_ids3))]

plt.figure(figsize=(10,10))
plt.imshow(bg, cmap="gray")
plt.scatter(d3["x"], d3["y"], s=0.1, facecolors="none", edgecolors="red")
plt.title(f"Tracked blobs overlay (round {r})")
plt.axis("off")
plt.show()

In [ ]:
r = 3
bg = merged_round_image(image_reg, r)
life = tracked_df.groupby("blob_id")["round"].nunique()

good_ids3 = life[life >= 0].index
d3 = tracked_df[(tracked_df["round"] == r) & (tracked_df["blob_id"].isin(good_ids3))]

plt.figure(figsize=(10,10))
plt.imshow(bg, cmap="gray")
plt.scatter(d3["x"], d3["y"], s=0.1, facecolors="none", edgecolors="red")
plt.title(f"Tracked blobs overlay (round {r})")
plt.axis("off")
plt.show()

In [ ]:
n_rounds = tracked_df["round"].nunique()
blob_ids_all_rounds = (
    tracked_df.groupby("blob_id")["round"]
              .nunique()
              .loc[lambda s: s == n_rounds]
              .index
)

In [ ]:
tracked_all_df = tracked_df[tracked_df["blob_id"].isin(blob_ids_all_rounds)].copy()
peaks_all_df = peaks_df[peaks_df["blob_id"].isin(blob_ids_all_rounds)].copy()
dominant_all_df = dominant_df[dominant_df["blob_id"].isin(blob_ids_all_rounds)].copy()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# df = your long table with columns: blob_id, round, channel, intensity_peak

# --- 1) per-blob per-round channel fraction of intensity
tot = peaks_all_df.groupby(["blob_id", "round"])["intensity_norm"].transform("sum")
df2 = peaks_all_df.copy()
df2["fraction"] = np.where(tot > 0, df2["intensity_norm"] / tot, np.nan)

# --- 2) summary per round/channel: mean fraction + SE
summary = (df2.groupby(["round", "channel"])["fraction"]
           .agg(mean="mean", std="std", n="count")
           .reset_index())
summary["se"] = summary["std"] / np.sqrt(summary["n"])
summary["fraction"] = summary["mean"]  # keep your plotting variable name
summary["round"] = summary["round"] + 3
summary = summary[(summary["round"] >= 3) & (summary["round"] <= 7)]

# channel → color mapping
ch_colors = {
    0: "green",     # G
    1: "red",       # T
    2: "magenta",   # A
    3: "cyan",      # C
}

ch_labels = {
    0: "G",
    1: "T",
    2: "A",
    3: "C",
}


plt.figure(figsize=(7, 5))

for ch in sorted(summary.channel.unique()):
    d = summary[summary.channel == ch].sort_values("round")
    plt.errorbar(
        d["round"],
        d["fraction"],
        yerr=d["se"],
        marker="o",
        capsize=3,
        color=ch_colors.get(ch, "gray"),
        label=ch_labels.get(ch, f"ch {ch}")   # <-- use G/T/A/C labels
    )

plt.xlabel("Round")
plt.ylabel("Mean intensity fraction")
plt.ylim(0, 1)
plt.legend(title="Base")   # optional: nicer title

rounds = np.sort(summary["round"].unique())
plt.xticks(rounds)

plt.tight_layout()
plt.savefig("mean_intensity_fraction_by_base_round3to7.svg",
            bbox_inches="tight")
plt.show()



In [ ]:
#Ithink i should aim to remove background pixels from the graph. Should I do this based on the calculated background used below
def compute_mean_background_per_channel(
    images,
    bg_y=(200, 250),
    bg_x=(500, 550)
):
    """
    images: (R, C, H, W)
    Returns DataFrame:
      channel, bg_mean
    """
    rows = []
    R, C, H, W = images.shape

    for ch in range(C):
        vals = []
        for r in range(R):
            img = images[r, ch]
            y0, y1 = bg_y
            x0, x1 = bg_x
            bg_pixels = img[y0:y1, x0:x1].ravel()
            if bg_pixels.size > 0:
                vals.append(bg_pixels.mean())
        rows.append({
            "channel": ch,
            "bg_mean_raw": float(np.mean(vals)),
            "bg_mediah_raw": float(np.median(vals))
        })

    return pd.DataFrame(rows)

bg_df = compute_mean_background_per_channel(image_reg)
print(bg_df)

In [ ]:
bg_df = bg_df.merge(
    ch_stats[["scale_hi"]],
    left_on="channel",
    right_index=True,
    how="left"
)

bg_df["bg_norm"] = bg_df["bg_mean_raw"] / bg_df["scale_hi"]
print(bg_df)


In [ ]:
peaks_bgsub = peaks_all_df.merge(
    bg_df[["channel", "bg_norm"]],
    on="channel",
    how="left"
)

peaks_bgsub["intensity_bgsub"] = (
    peaks_bgsub["intensity_norm"] - peaks_bgsub["bg_norm"]
)

peaks_bgsub["intensity_bgsub"] = peaks_bgsub["intensity_bgsub"].clip(lower=0)


In [ ]:
# background should sit near zero now
peaks_bgsub.groupby("channel")["intensity_bgsub"].quantile([0.05, 0.1, 0.5])

# nothing negative
assert (peaks_bgsub["intensity_bgsub"] >= 0).all()


In [ ]:
peaks_bgsub

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# df = your long table with columns: blob_id, round, channel, intensity_peak

# --- 1) per-blob per-round channel fraction of intensity
tot = peaks_bgsub.groupby(["blob_id", "round"])["intensity_bgsub"].transform("sum")
df2 = peaks_bgsub.copy()
df2["fraction"] = np.where(tot > 0, df2["intensity_bgsub"] / tot, np.nan)

# --- 2) summary per round/channel: mean fraction + SE
summary = (df2.groupby(["round", "channel"])["fraction"]
           .agg(mean="mean", std="std", n="count")
           .reset_index())
summary["se"] = summary["std"] / np.sqrt(summary["n"])
summary["fraction"] = summary["mean"]  # keep your plotting variable name
summary["round"] = summary["round"] + 3
summary = summary[(summary["round"] >= 3) & (summary["round"] <= 7)]

# channel → color mapping
ch_colors = {
    0: "green",     # G
    1: "red",       # T
    2: "magenta",   # A
    3: "cyan",      # C
}

ch_labels = {
    0: "G",
    1: "T",
    2: "A",
    3: "C",
}


plt.figure(figsize=(7, 5))

for ch in sorted(summary.channel.unique()):
    d = summary[summary.channel == ch].sort_values("round")
    plt.errorbar(
        d["round"],
        d["fraction"],
        yerr=d["se"],
        marker="o",
        capsize=3,
        color=ch_colors.get(ch, "gray"),
        label=ch_labels.get(ch, f"ch {ch}")   # <-- use G/T/A/C labels
    )

plt.xlabel("Round")
plt.ylabel("Mean intensity fraction")
plt.ylim(0, 1)
plt.legend(title="Base")   # optional: nicer title

rounds = np.sort(summary["round"].unique())
plt.xticks(rounds)

plt.tight_layout()
plt.savefig("mean_intensity_fraction_by_base_round3to7.svg",
            bbox_inches="tight")
plt.show()

